# 13 — Optimization Beyond Gradient Descent Practice

This notebook practices gradient descent, momentum, RMSProp, Adam, gradient clipping, learning rate schedules, and optimization diagnostics.

In [ ]:
import numpy as np

## 1. Objective and Gradient

$$
J(\theta_1,\theta_2)=0.08\theta_1^2+2.5\theta_2^2
$$

In [ ]:
def objective(theta):
    x, y = theta
    return 0.08 * x ** 2 + 2.5 * y ** 2


def gradient(theta):
    x, y = theta
    return np.array([0.16 * x, 5.0 * y])

## 2. Plain Gradient Descent

$$
\theta_{t+1}=\theta_t-\alpha g_t
$$

In [ ]:
def gradient_descent(start, lr=0.1, steps=80):
    theta = np.array(start, dtype=float)
    losses = []
    for _ in range(steps):
        losses.append(objective(theta))
        theta = theta - lr * gradient(theta)
    return theta, np.array(losses)


theta_gd, losses_gd = gradient_descent([5.5, 2.5], lr=0.18)

theta_gd, losses_gd[-1]

## 3. Momentum

$$
v_t=\beta v_{t-1}+g_t
$$

$$
\theta_{t+1}=\theta_t-\alpha v_t
$$

In [ ]:
def momentum_descent(start, lr=0.1, beta=0.9, steps=80):
    theta = np.array(start, dtype=float)
    velocity = np.zeros_like(theta)
    losses = []
    for _ in range(steps):
        losses.append(objective(theta))
        grad = gradient(theta)
        velocity = beta * velocity + grad
        theta = theta - lr * velocity
    return theta, np.array(losses)


theta_momentum, losses_momentum = momentum_descent([5.5, 2.5], lr=0.18, beta=0.82)

theta_momentum, losses_momentum[-1]

## 4. RMSProp and Adam

In [ ]:
def rmsprop_descent(start, lr=0.05, beta=0.9, eps=1e-8, steps=80):
    theta = np.array(start, dtype=float)
    square_avg = np.zeros_like(theta)
    losses = []
    for _ in range(steps):
        losses.append(objective(theta))
        grad = gradient(theta)
        square_avg = beta * square_avg + (1 - beta) * (grad ** 2)
        theta = theta - lr * grad / (np.sqrt(square_avg) + eps)
    return theta, np.array(losses)


def adam_descent(start, lr=0.08, beta1=0.9, beta2=0.999, eps=1e-8, steps=80):
    theta = np.array(start, dtype=float)
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)
    losses = []
    for step in range(1, steps + 1):
        losses.append(objective(theta))
        grad = gradient(theta)
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad ** 2)
        m_hat = m / (1 - beta1 ** step)
        v_hat = v / (1 - beta2 ** step)
        theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
    return theta, np.array(losses)


theta_rms, losses_rms = rmsprop_descent([5.5, 2.5], lr=0.08)
theta_adam, losses_adam = adam_descent([5.5, 2.5], lr=0.12)

losses_rms[-1], losses_adam[-1]

## 5. Gradient Clipping

$$
g_{clipped}=g\cdot\min\left(1,\frac{c}{\|g\|}\right)
$$

In [ ]:
def clip_gradient_by_norm(grad, max_norm):
    norm = np.linalg.norm(grad)
    if norm <= max_norm:
        return grad
    return grad * (max_norm / norm)

large_grad = np.array([3.0, 12.0])
clipped = clip_gradient_by_norm(large_grad, max_norm=5.0)

np.linalg.norm(large_grad), clipped, np.linalg.norm(clipped)

## 6. Learning Rate Schedule

In [ ]:
def step_decay_lr(initial_lr, step, drop_every=25, drop_factor=0.5):
    num_drops = step // drop_every
    return initial_lr * (drop_factor ** num_drops)

[step_decay_lr(0.1, step) for step in [0, 10, 25, 40, 50, 75]]

## Reflection

Optimization beyond gradient descent is about controlling parameter movement: momentum remembers direction, adaptive methods rescale gradients, schedules control training rhythm, and clipping prevents unstable updates.